# RAG Demo (Local Ollama + Chroma) — Sequential / No Functions

A **straight-line** walkthrough of local RAG, written for live teaching: read it top
to bottom, run one cell at a time, and explain each step as you go. There are **no
helper functions to jump between** — every step is right where it runs.

**The whole idea of RAG, in one line:** retrieve the most relevant chunks of your
documents, paste them into the prompt as *context*, and tell the LLM to answer
**only** from that context (with citations).

**Pipeline (one cell each):**
`config → list books → download → chunk → embed → store → retrieve → build context → ask the LLM`

**Prereqs:** Ollama running, with a chat model and an embedding model pulled:
```bash
ollama list
ollama pull llama3.1:8b
ollama pull embeddinggemma
```

The embed step is the slow one, so we run it **once**: Chroma keeps the vectors on
disk, and a single `if not already_built:` guard skips the rebuild on every later run.

In [8]:
# Run once if needed:
# %pip install -q chromadb requests tqdm numpy

In [9]:
import os, re, json, textwrap
from pathlib import Path

import numpy as np
import requests
import chromadb

# Progress bar for the one-time embedding step (optional).
try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

## 1. Configuration

Everything you might tweak lives in one place: which models to call, how big the
chunks are, how many to retrieve, and where things are stored on disk.

In [10]:
OLLAMA_URL  = "http://localhost:11434"
CHAT_MODEL  = "llama3.1:8b"
EMBED_MODEL = "embeddinggemma:latest"        # e.g. "nomic-embed-text"

WORDS_PER_CHUNK = 300      # size of each chunk, in words
OVERLAP_WORDS   = 60       # chunks overlap so we don't slice a thought in half
TOPK            = 5        # how many chunks to retrieve per question

CORPUS_DIR        = Path("corpus_jupyter")   # downloaded .txt books land here
CHROMA_PATH       = "./chroma_db"            # Chroma persists vectors here
CHROMA_COLLECTION = "rag_demo"

EMBED_BATCH_SIZE  = 64
DOWNLOAD_FROM_WEB = True    # False = stay offline, use whatever .txt is already in CORPUS_DIR

## 2. The corpus — 13 books from Project Gutenberg

These are our "documents." Set `DOWNLOAD_FROM_WEB = False` to stay offline (then
drop your own `.txt` files into `CORPUS_DIR`).

In [11]:
GUTENBERG_BOOKS = {
    "Moby-Dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
    "Pride and Prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
    "Frankenstein": "https://www.gutenberg.org/files/84/84-0.txt",
    "Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "Dracula": "https://www.gutenberg.org/files/345/345-0.txt",
    "A Tale of Two Cities": "https://www.gutenberg.org/files/98/98-0.txt",
    "The Great Gatsby": "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
    "Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "War and Peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
    "Jane Eyre": "https://www.gutenberg.org/files/1260/1260-0.txt",
    "The Picture of Dorian Gray": "https://www.gutenberg.org/files/174/174-0.txt",
    "Crime and Punishment": "https://www.gutenberg.org/files/2554/2554-0.txt",
    "Wuthering Heights": "https://www.gutenberg.org/files/768/768-0.txt",
}
print(f"{len(GUTENBERG_BOOKS)} books in the corpus.")

13 books in the corpus.


## 3. Talk to Ollama

One `requests.Session` with `trust_env = False` so the localhost call isn't routed
through a VPN/proxy. We hit Ollama's HTTP API directly:

- **embeddings** → `POST /api/embed`  (turns text into vectors)
- **chat**       → `POST /api/chat`   (the LLM that writes the answer)

Quick reachability check below.

In [12]:
SESSION = requests.Session()
SESSION.trust_env = False

# Sanity check: is Ollama up, and are our two models pulled?
r = SESSION.get(f"{OLLAMA_URL}/api/tags", timeout=30)
r.raise_for_status()
available = [m.get("name", "") for m in r.json().get("models", [])]
print(f"Ollama reachable — {len(available)} model(s) installed.")
for name in (CHAT_MODEL, EMBED_MODEL):
    print(f"  {name}: {'ok' if name in available else 'MISSING — run `ollama pull`'}")

Ollama reachable — 12 model(s) installed.
  llama3.1:8b: ok
  embeddinggemma:latest: ok


## 4. Open the vector store (and decide whether we must build it)

Chroma is our vector database. It **persists to disk** at `CHROMA_PATH`, so the
expensive embedding work only has to happen once. We open the collection and check
whether it already holds vectors — that single boolean, `already_built`, is our
entire caching strategy.

In [13]:
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(
    CHROMA_COLLECTION, metadata={"hnsw:space": "cosine"}   # cosine distance: lower = closer
)

already_built = collection.count() > 0
print(f"Chroma collection '{CHROMA_COLLECTION}': {collection.count()} vectors.")
print("Vectors already present — the embed step will be skipped." if already_built
      else "Empty collection — we'll embed and populate it below.")

Chroma collection 'rag_demo': 8509 vectors.
Vectors already present — the embed step will be skipped.


## 5. Download (or load) the books

For each title: if we already have the `.txt` on disk, just read it. Otherwise
download it and strip Project Gutenberg's license header/footer so only the actual
book text remains. This is cheap, so we run it every time.

In [14]:
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

# Project Gutenberg wraps each book in license boilerplate; keep only what's between these.
START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

docs = []
for title, url in GUTENBERG_BOOKS.items():
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")    # filesystem-safe filename
    path = CORPUS_DIR / f"{safe_name}.txt"

    if not path.exists():
        if not DOWNLOAD_FROM_WEB:
            continue
        print(f"Downloading: {title}")
        resp = SESSION.get(url, timeout=180)
        resp.raise_for_status()
        raw = resp.text
        start, end = START_MARK.search(raw), END_MARK.search(raw)
        if start and end and end.start() > start.end():
            raw = raw[start.end():end.start()]
        path.write_text(raw.strip(), encoding="utf-8")

    docs.append({"title": title, "url": url,
                 "text": path.read_text(encoding="utf-8", errors="ignore")})

print(f"Loaded {len(docs)} books.")

Loaded 13 books.


## 6. Chunk each book into overlapping windows

LLMs and embedding models work best on small passages, and we want retrieval to
return *focused* pieces, not whole books. We slide a `WORDS_PER_CHUNK`-word window
across each book, stepping forward by `WORDS_PER_CHUNK - OVERLAP_WORDS` words so
consecutive chunks overlap and we never cut a thought cleanly in half.

Each chunk carries an id like `pride-and-prejudice#23` that we'll use for citations.

In [15]:
chunks = []
for d in docs:
    doc_id = re.sub(r"[^a-z0-9]+", "-", d["title"].lower()).strip("-")   # "pride-and-prejudice"

    words = re.sub(r"\s+", " ", d["text"]).strip().split()
    step = max(1, WORDS_PER_CHUNK - OVERLAP_WORDS)

    i = 0
    for start in range(0, len(words), step):
        window = words[start:start + WORDS_PER_CHUNK]
        if len(window) < max(60, WORDS_PER_CHUNK // 4):    # drop a tiny trailing scrap
            break
        chunks.append({
            "id": f"{doc_id}#{i}",
            "doc_id": doc_id,
            "chunk_index": i,
            "title": d["title"],
            "source": d["url"],
            "text": " ".join(window),
        })
        i += 1
        if start + WORDS_PER_CHUNK >= len(words):
            break

print(f"Built {len(chunks)} chunks from {len(docs)} books.\n")
print("Example chunk:")
print(f"  id   : {chunks[0]['id']}")
print(f"  words: {len(chunks[0]['text'].split())}")
print("  text :", textwrap.fill(chunks[0]['text'][:300] + " ...", width=80,
                                subsequent_indent="         "))

Built 8509 chunks from 13 books.

Example chunk:
  id   : moby-dick#0
  words: 300
  text : MOBY-DICK; or, THE WHALE. By Herman Melville CONTENTS ETYMOLOGY. EXTRACTS
         (Supplied by a Sub-Sub-Librarian). CHAPTER 1. Loomings. CHAPTER 2. The
         Carpet-Bag. CHAPTER 3. The Spouter-Inn. CHAPTER 4. The Counterpane.
         CHAPTER 5. Breakfast. CHAPTER 6. The Street. CHAPTER 7. The Chapel.
         CHAPTER 8. The Pulp ...


## 7. Embed every chunk and store it in Chroma  *(the expensive, cached step)*

This is the one slow step: we send each batch of chunk texts to the embedding model,
get back a vector per chunk, and hand the vectors to Chroma. Because Chroma persists
to disk, the whole thing is guarded by `if not already_built:` — run it once, and
every future run skips straight to retrieval.

*(We let Chroma's cosine space handle normalization, so there's no separate
normalize step — cosine ranking doesn't care about vector length.)*

In [16]:
if already_built:
    print(f"Skipping embed — Chroma already has {collection.count()} vectors.")
else:
    batches = range(0, len(chunks), EMBED_BATCH_SIZE)
    if tqdm:
        batches = tqdm(batches, desc=f"Embedding with {EMBED_MODEL}")

    for s in batches:
        batch = chunks[s:s + EMBED_BATCH_SIZE]
        texts = [c["text"] for c in batch]

        # One POST returns one vector per text.
        resp = SESSION.post(f"{OLLAMA_URL}/api/embed",
                            json={"model": EMBED_MODEL, "input": texts}, timeout=600)
        resp.raise_for_status()
        vectors = resp.json()["embeddings"]

        collection.add(
            ids=[c["id"] for c in batch],
            documents=texts,
            metadatas=[{"doc_id": c["doc_id"], "chunk_index": c["chunk_index"],
                        "title": c["title"], "source": c["source"]} for c in batch],
            embeddings=vectors,
        )

    print(f"Done. Chroma now holds {collection.count()} vectors.")

Skipping embed — Chroma already has 8509 vectors.


## 8. Retrieval — find the chunks closest to a question

Embed the question with the **same** model, then ask Chroma for the `TOPK` nearest
chunks by cosine distance (lower = closer). This is the "R" in RAG — no LLM yet,
just a similarity search.

In [17]:
question = "Who is Elizabeth Bennet?"

# Embed the question (same endpoint as before, just one text).
resp = SESSION.post(f"{OLLAMA_URL}/api/embed",
                    json={"model": EMBED_MODEL, "input": [question]}, timeout=600)
resp.raise_for_status()
query_vector = resp.json()["embeddings"][0]

# Ask Chroma for the nearest chunks. We sent one query, so results live at index [0].
res = collection.query(query_embeddings=[query_vector], n_results=TOPK,
                       include=["documents", "metadatas", "distances"])
docs_out  = res["documents"][0]
metas_out = res["metadatas"][0]
dists_out = res["distances"][0]

print(f"Top {TOPK} chunks for: {question!r}\n")
for meta, dist in zip(metas_out, dists_out):
    print(f"  distance={dist:.4f}  [{meta['doc_id']}#{meta['chunk_index']}]  {meta['title']}")

Top 5 chunks for: 'Who is Elizabeth Bennet?'

  distance=0.4757  [pride-and-prejudice#23]  Pride and Prejudice
  distance=0.5044  [pride-and-prejudice#490]  Pride and Prejudice
  distance=0.5237  [pride-and-prejudice#482]  Pride and Prejudice
  distance=0.5250  [pride-and-prejudice#486]  Pride and Prejudice
  distance=0.5263  [pride-and-prejudice#34]  Pride and Prejudice


## 9. Answering — stuff the context into the prompt and ask the LLM

**This is the whole point of RAG.** We paste the retrieved chunks into the prompt as
*context*, add the question, and instruct the LLM to answer **only** from that
context and cite its sources.

This cell is **self-contained** — it retrieves *and* answers — so you can copy it,
change `question`, and re-run to ask the knowledge base anything.

In [18]:
question = "How does Victor Frankenstein create life?"

# --- Retrieve: embed the question, find the nearest chunks --------------------
resp = SESSION.post(f"{OLLAMA_URL}/api/embed",
                    json={"model": EMBED_MODEL, "input": [question]}, timeout=600)
resp.raise_for_status()
query_vector = resp.json()["embeddings"][0]

res = collection.query(query_embeddings=[query_vector], n_results=TOPK,
                       include=["documents", "metadatas", "distances"])
hits = list(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]))

# --- Build the context block: a [doc_id#chunk] tag before each chunk ----------
context = ""
for text, meta, dist in hits:
    context += f"[{meta['doc_id']}#{meta['chunk_index']}] {meta['title']}\n{text}\n---\n"

# --- Ask the LLM to answer ONLY from that context -----------------------------
system_prompt = (
    "You are a helpful assistant. Answer ONLY using the provided context. "
    "If the answer is not in the context, say: 'I don't know based on the provided context.' "
    "Cite sources in square brackets like [doc_id#chunk_index] for each key claim."
)
user_prompt = f"Question: {question}\n\nContext:\n{context}"

resp = SESSION.post(f"{OLLAMA_URL}/api/chat",
                    json={"model": CHAT_MODEL, "stream": False,
                          "options": {"temperature": 0.2},
                          "messages": [{"role": "system", "content": system_prompt},
                                       {"role": "user", "content": user_prompt}]},
                    timeout=600)
resp.raise_for_status()
answer = resp.json()["message"]["content"]

# --- Show it ------------------------------------------------------------------
print("[Q]", question, "\n")
print("[ANSWER]\n")
print(textwrap.fill(answer.strip(), width=100))
print("\n[SOURCES RETRIEVED]")
for text, meta, dist in hits:
    print(f"  distance={dist:.4f}  [{meta['doc_id']}#{meta['chunk_index']}]  {meta['title']}")

[Q] How does Victor Frankenstein create life? 

[ANSWER]

According to the provided context, Victor Frankenstein creates life by infusing it into an inanimate
body. He worked hard for nearly two years, depriving himself of rest and health, to achieve this
goal [frankenstein#62]. He used his knowledge and skills to collect and arrange materials, and then
began the process of creating a human being [frankenstein#57]. Frankenstein's goal was to bestow
animation upon lifeless matter, and he was driven by a desire to create a new species and be hailed
as its creator [frankenstein#57]. He was able to create a being of gigantic stature, about eight
feet in height, with a complex and wonderful organization [frankenstein#56].

[SOURCES RETRIEVED]
  distance=0.5906  [frankenstein#62]  Frankenstein
  distance=0.6023  [frankenstein#127]  Frankenstein
  distance=0.6088  [frankenstein#56]  Frankenstein
  distance=0.6146  [frankenstein#305]  Frankenstein
  distance=0.6169  [frankenstein#57]  Frankens

## 10. Maintenance — wipe and rebuild from scratch

Re-running the notebook reuses the existing Chroma vectors. To force a clean rebuild
(e.g. after changing the chunk size or the embedding model), delete the collection
and re-run from the top. Uncomment to use.

In [19]:
# client.delete_collection(CHROMA_COLLECTION)
# print("Deleted collection — re-run the notebook from the top to rebuild.")